# 📌 Agent Skills: how a skill works behind the scenes

![Topic](https://img.shields.io/badge/Topic-Agent_Skills-blue?style=flat-square)
![Category](https://img.shields.io/badge/Category-Agentic-blueviolet?style=flat-square)
![Level](https://img.shields.io/badge/Level-Intermediate-yellow?style=flat-square)
![Last Updated](https://img.shields.io/badge/Updated-November%202026-blue?style=flat-square)
<br>
<br>
<br>
> <span style="font-size:20px;">**TL;DR** — A skill is a folder with a SKILL.md file, plus optional scripts and reference files. The agent’s harness (the program that runs the model’s loop) puts only each skill’s name and description in the prompt. The model decides from those descriptions whether to open a skill, and the full instructions then enter the context only if it does. Scripts are executed rather than read, so only their output costs tokens. A skill is therefore a context-management mechanism as much as a file format.</span>

---
## 1. Overview

<!-- What is this concept? 2–4 sentences that a newcomer could understand.
     Include: what problem it solves, why it exists, where it fits in the AI landscape. -->

**Problem.** A general agent needs procedural know-how (how to fill a PDF, write a changelog, review SQL). Putting everything in the system prompt wastes context, and re-explaining it every session is fragile. Anthropic's answer, introduced in October 2025, is organized folders of instructions, scripts and resources that agents discover and load dynamically; the article compares writing one to building an onboarding guide for a new hire (Anthropic, 2025).

**What a skill is.** At minimum, a directory containing a `SKILL.md` file, with YAML frontmatter followed by Markdown (agentskills.io, 2026).

```
pdf-processing/
├── SKILL.md          # required: frontmatter (name, description) + instructions
├── scripts/          # optional: code the agent runs
├── references/       # optional: docs loaded only if SKILL.md says so
└── assets/           # optional: templates, schemas
```

`name` is limited to 64 characters of lowercase letters, digits and hyphens. It must not start or end with a hyphen, must not contain `--`, and must match the folder name. `description` is limited to 1,024 characters and should say both what the skill does and when to use it. The spec also defines optional fields such as `license`, `compatibility`, `metadata` and `allowed-tools` (agentskills.io, 2026).

**Where it sits.**

| Term | What it carries | Question it answers |
|------|-----------------|---------------------|
| **Skill** | Procedure: how to do a task | "How should I do this?" |
| **MCP server** | Access to tools and data | "What can I reach?" |
| **Agent.md** | Project facts, always loaded | "What is this project?" |
| **Progressive disclosure** | The loading strategy of a skill | "How much do I load, and when?" |

Anthropic plans to explore how Skills can complement MCP servers by teaching agents more complex workflows that involve external tools (Anthropic, 2025). The format was published as an open standard on 18 December 2025. Developer tools adopted it first: GitHub Copilot added support the same day, and VS Code followed in January 2026 (GitHub, 2025). By October 2026 more than 70 agent platforms, including ChatGPT, Cursor, GitHub Copilot and VS Code, had integrated it (Anthropic, 2026). Consumer products came later: ChatGPT (July 2026), the Gemini app, where Skills replace Gems (September 2026), and Microsoft 365 Copilot (October 2026). Jisc notes that OpenAI and Microsoft document the file format without referencing the standard itself (Jisc, 2026). On 22 October 2026 Anthropic donated the standard to the Agentic AI Foundation (AAIF) under the Linux Foundation, which also hosts MCP (Anthropic, 2026).

---
## 2. How It Works

<!-- Break the concept into numbered steps or subsections.
     Use diagrams (images from assets/) where helpful.
     Each subsection should follow: description → danger/difficulty level → counter-technique or note -->

A skill adds no new model capability. The harness reads files and the model chooses what to read next. Three levels of detail are loaded on demand.

| Level | What is loaded | When | Size guidance |
|-------|----------------|------|---------------|
| **1. Metadata** | `name` + `description` of every installed skill | At startup, always | About 100 tokens per skill |
| **2. Instructions** | Body of `SKILL.md` | When the model judges the skill relevant | Under 5,000 tokens recommended (under 500 lines) |
| **3. Resources** | Linked files (`references/`) and script output | Only if the body says so and the task needs it | Effectively unbounded |

(Size guidance: agentskills.io, 2026.)

### 2.1 Startup: the harness scans, it does not read everything
At startup the agent pre-loads the `name` and `description` of every installed skill into its system prompt (Anthropic, 2025). The bodies stay on disk.

### 2.2 Triggering: the model decides, not a router
For automatic triggering, nothing in the harness matches the request to a skill. The model reads the descriptions in its prompt and decides, as it would choose a tool. Some harnesses also let the user invoke a skill explicitly by name (e.g. `/skill-name` in Claude Code); the harness then loads it directly. Anthropic therefore advises paying special attention to `name` and `description`, since the model uses them to decide whether to trigger the skill (Anthropic, 2025). The description is the whole "retrieval index".

### 2.3 Loading: activation is an ordinary tool call
In Anthropic's PDF example, the model invokes a Bash tool to read `pdf/SKILL.md`, then reads the bundled `forms.md`, and only then proceeds with the task (Anthropic, 2025). Activation there is an ordinary tool call whose result lands in the context. The spec does not prescribe the mechanism: Claude Code, for instance, exposes a dedicated Skill tool instead of a raw file read.

### 2.4 Scripts: code is executed, not read
The PDF skill ships a Python script that extracts form fields, and the model can run it without loading the script or the PDF into context (Anthropic, 2025). Only stdout comes back. This saves tokens and makes the result deterministic.

### 2.5 One turn, end to end

```mermaid
flowchart LR
    A[Startup<br/>harness scans SKILL.md files] --> B[System prompt<br/>name + description x N]
    B --> C[User request]
    C --> D{Model compares request<br/>to descriptions}
    D -->|no match| E[Answers from own knowledge]
    D -->|match| F[Reads SKILL.md<br/>level 2]
    F --> G[Reads linked files<br/>level 3, if needed]
    F --> H[Runs script<br/>only stdout returns]
    G --> I[Answer]
    H --> I

    classDef node fill:#ADEBB3,stroke:#00BB77,color:#1a1a1a
    classDef final fill:#ADEBB3,stroke:#00BB77,color:#1a1a1a,font-weight:bold

    class A,B,C,D,E,F,G,H node
    class I final
```

### 2.6 Who does what

| Harness (deterministic code) | Model (probabilistic) |
|------------------------------|-----------------------|
| Finds skill folders, parses frontmatter | Judges which description matches the task |
| Injects metadata into the prompt | Decides to open `SKILL.md` and which linked files to read |
| Provides file-read and code-execution tools | Decides when to run a script and how to use its output |
| Applies permissions, e.g. pre-approving the tools listed in `allowed-tools` (experimental, support varies) | Follows (or drifts from) the instructions |

---
## 3. Advantages & Limitations

| | Aspect | Commentary |
|--|--------|------------|
| 🟢 | **Context economy vs. a long system prompt.** | Only metadata is always present; bodies and resources come on demand. In the toy: 53 tokens at startup versus 172 with every skill file inlined (toy numbers, not a benchmark). |
| 🟢 | **Effectively unbounded bundled knowledge.** | Because the agent does not have to read a whole skill into context, the amount of content a skill can bundle is effectively unbounded (Anthropic, 2025). |
| 🟢 | **Deterministic steps via scripts vs. generating everything with tokens.** | Sorting a list through token generation costs far more than running a sorting algorithm, and code gives the reliability many applications require (Anthropic, 2025). In the toy, the script returns 4 tokens of output instead of the model reading the 5,001-token CSV. |
| 🟢 | **Portability thanks to an open standard.** | The same folder works across vendors that implemented the spec (see section 1). Since October 2026 Agent Skills has been governed by the Agentic AI Foundation, which also hosts MCP and AGENTS.md (Anthropic, 2026; Agentailor, 2026). |
| 🟢 | **Simple, shareable, versionable.** | It is plain files: a skill can be reviewed, diffed and stored in git like code, with no training or fine-tuning loop. |
| 🔴 | **Triggering rests on the quality of a description.** | The decision is made by the model from a short text, with no guarantee. A vague or overlapping description means a skill that never fires, or fires wrongly. This is my analysis, though Anthropic's advice to watch the `description` points the same way. |
| 🔴 | **Security: a skill is instructions plus code.** | Anthropic warns that malicious skills may introduce vulnerabilities or direct the agent to exfiltrate data, and recommends installing only from trusted sources and auditing bundled scripts (Anthropic, 2025). Pillar Security research (July 2026) showed sandbox escapes across several coding agents, so isolation is not a given either (Developers Digest, 2026). |
| 🔴 | **Needs a capable harness.** | Anthropic's description relies on an agent with filesystem and code-execution tools (Anthropic, 2025). In a plain chat API without those tools, levels 2 and 3 do not work as described. |
| 🔴 | **Metadata cost grows with the number of skills.** | About 100 tokens each is small, but with hundreds of skills the always-loaded block grows, and the model has more near-duplicate descriptions to choose from. Analysis, not a measured result. |
| 🔴 | **A format, not a quality guarantee.** | One 2026 paper notes the standard defines a format without addressing governance constraints or validators for institutional knowledge at enterprise scale (Knowledge Activation, 2026). Evaluating whether a skill really improves results is up to you. |

---
## 4. Code Example

> **Goal:** Make the mechanism visible: scan frontmatter only, pick a skill, load the body, run a script, and count context size at each step. **No real API calls.** The "model" is mocked with keyword overlap, and tokens are approximated as words.

In [1]:
# ---------------------------------------------------------------------------
# Toy "skills harness": shows WHAT the harness does, WHAT the model decides,
# and WHAT actually lands in the context window. No LLM, no API, stdlib only.
# ---------------------------------------------------------------------------
import re, subprocess, sys, tempfile, textwrap
from pathlib import Path

# ---- 0. Build a fake skills folder on disk (real files, like in a repo) ----
root = Path(tempfile.mkdtemp()) / "skills"

def write(path, content):
    p = root / path
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(textwrap.dedent(content).lstrip())

write("csv-stats/SKILL.md", """\
    ---
    name: csv-stats
    description: Compute mean, min and max of a numeric CSV column. Use when the user asks for statistics on a CSV file.
    ---
    # CSV stats
    1. Run `scripts/stats.py <file> <column>`; do NOT read the CSV yourself.
    2. Report the printed result.
    3. If the user asks about outliers, read `references/outliers.md` first.
    """)
write("csv-stats/references/outliers.md", "Outlier policy: flag values beyond 3 standard deviations.\n")
write("csv-stats/scripts/stats.py", """\
    import csv, sys
    path, col = sys.argv[1], sys.argv[2]
    vals = [float(r[col]) for r in csv.DictReader(open(path))]
    print(f"n={len(vals)} mean={sum(vals)/len(vals):.2f} min={min(vals)} max={max(vals)}")
    """)
write("release-notes/SKILL.md", """\
    ---
    name: release-notes
    description: Draft release notes from a list of commits. Use when the user wants a changelog or release notes.
    ---
    # Release notes
    Group commits into: New features, Bug fixes, Known issues. ...(300 more lines)...
    """)

# Big data file the model should never have to read
data = root / "sales.csv"
data.write_text("amount\n" + "\n".join(str(i) for i in range(1, 5001)))

def tokens(text):            # crude proxy: 1 token ~ 1 word
    return len(text.split())

# ---- 1. HARNESS, at startup: read ONLY the frontmatter of every skill ------
def scan_skills(root):
    registry = {}
    for md in root.glob("*/SKILL.md"):
        front = re.match(r"---\n(.*?)\n---", md.read_text(), re.S).group(1)
        meta = dict(line.split(": ", 1) for line in front.splitlines())
        registry[meta["name"]] = {"description": meta["description"], "path": md}
    return registry

registry = scan_skills(root)
system_prompt = "You are an agent. Available skills:\n" + "\n".join(
    f"- {n}: {s['description']} (file: {s['path']})" for n, s in registry.items())

context = [("system prompt + skill metadata", system_prompt)]

def show(step):
    total = sum(tokens(t) for _, t in context)
    print(f"[{step:<34}] context = {total:>3} tokens   ({len(context)} blocks)")

show("startup (level 1: metadata only)")

# ---- 2. MODEL decision (mocked): which skill matches the request? ---------
def mock_model_pick_skill(request, registry):
    """Stand-in for the LLM: word overlap between request and descriptions.
    In reality this is the model reading the metadata in its prompt. There is
    no router, no embedding index, no rule engine."""
    stop = {"the", "of", "a", "an", "to", "is", "what", "in", "my", "i", "on", "for",
            "use", "when", "user", "from", "and", "or", "asks", "wants"}
    tok = lambda t: set(re.findall(r"\w+", t.lower())) - stop
    words = tok(request)
    scores = {n: len(words & tok(s["description"])) for n, s in registry.items()}
    best = max(scores, key=scores.get)
    return best if scores[best] > 1 else None

# ---- 3. Tools the harness exposes to the model -----------------------------
def tool_read_file(path):                       # output enters the context
    text = Path(path).read_text()
    context.append((f"read_file({Path(path).name})", text))
    return text

def tool_run_script(path, *args):               # only STDOUT enters the context
    out = subprocess.run([sys.executable, str(path), *args],
                         capture_output=True, text=True).stdout.strip()
    context.append((f"run_script({Path(path).name}) stdout", out))
    return out

# ---- 4. One agent turn ------------------------------------------------------
def agent_turn(request):
    print(f"\nUSER: {request!r}")
    context.append(("user message", request)); show("user message arrives")

    skill = mock_model_pick_skill(request, registry)
    if skill is None:
        print("  model: no skill matches -> answers from its own knowledge")
        return
    print(f"  model: description of '{skill}' matches -> opens its SKILL.md")

    tool_read_file(registry[skill]["path"]);            show("level 2: SKILL.md body loaded")

    # The body tells the model what to do next (instructions, not code):
    skill_dir = registry[skill]["path"].parent
    if skill == "csv-stats":
        if "outlier" in request.lower():
            tool_read_file(skill_dir / "references" / "outliers.md"); show("level 3: outliers.md loaded")
        result = tool_run_script(skill_dir / "scripts" / "stats.py", str(data), "amount")
        show("script ran (only stdout kept)")
        print("  model answer based on:", result)

agent_turn("What is the average of the amount column in my CSV file? I need statistics.")
agent_turn("Any outliers in the amount column of my CSV file? I need statistics.")
agent_turn("What is the capital of France?")

# ---- 5. Baselines: what the context would hold without skills --------------
# (a) Lazy loading: inline every skill file (not the user's data) at startup.
skill_files = [p for p in root.glob("*/**/*") if p.is_file()]
everything = system_prompt + "".join(p.read_text() for p in skill_files)
print(f"\nBaseline 'inline every skill file': {tokens(everything):,} tokens "
      f"vs. {tokens(system_prompt):,} at startup with skills")
# (b) Scripts: the model reads the CSV itself instead of running stats.py.
stdout = next(t for label, t in context if label.startswith("run_script"))
print(f"Baseline 'model reads sales.csv itself': {tokens(data.read_text()):,} tokens "
      f"vs. {tokens(stdout):,} for the script's stdout")

[startup (level 1: metadata only)  ] context =  53 tokens   (1 blocks)

USER: 'What is the average of the amount column in my CSV file? I need statistics.'
[user message arrives              ] context =  68 tokens   (2 blocks)
  model: description of 'csv-stats' matches -> opens its SKILL.md
[level 2: SKILL.md body loaded     ] context = 123 tokens   (3 blocks)
[script ran (only stdout kept)     ] context = 127 tokens   (4 blocks)
  model answer based on: n=5000 mean=2500.50 min=1.0 max=5000.0

USER: 'Any outliers in the amount column of my CSV file? I need statistics.'
[user message arrives              ] context = 140 tokens   (5 blocks)
  model: description of 'csv-stats' matches -> opens its SKILL.md
[level 2: SKILL.md body loaded     ] context = 195 tokens   (6 blocks)
[level 3: outliers.md loaded       ] context = 203 tokens   (7 blocks)
[script ran (only stdout kept)     ] context = 207 tokens   (8 blocks)
  model answer based on: n=5000 mean=2500.50 min=1.0 max=5000.0

USER: 'W

---
## 5. Key Takeaways
<div style="font-size: 16px; line-height: 1.6;">

- **A skill is a folder with a `SKILL.md`, and its power is lazy loading.** The harness injects only name and description; the body and extra files enter the context if the model asks for them.
- **There is no router for automatic triggering.** The model matches the request against descriptions, so the description is the interface and the weakest link.
- **Activation is an ordinary tool call (a file read in Anthropic's example, a dedicated Skill tool in Claude Code), and scripts return only their output,** which makes parts of the workflow deterministic and cheap.
- **The costs are security, reliability of triggering and dependence on the harness.** Treat a skill like code you run: audit it, and evaluate it on representative tasks.
- **Skills are context engineering in file form.** They connect to the MCP notebook (tools versus procedures) and set up the planned harness and security topics.

</div>

## 6. Sources 

**Primary**

Zhang, B., Lazuka, K. & Murag, M. (2025). *Equipping agents for the real world with Agent Skills.* Anthropic Engineering. https://www.anthropic.com/engineering/equipping-agents-for-the-real-world-with-agent-skills (read in full; page modified October 2026)

Anthropic. (2026). *Donating Agent Skills to the Agentic AI Foundation.* https://www.anthropic.com/news/donating-agent-skills-to-the-agentic-ai-foundation

GitHub. (2025). *GitHub Copilot now supports Agent Skills.* GitHub Changelog, 18 December 2025. https://github.blog/changelog/2025-12-18-github-copilot-now-supports-agent-skills/

Agent Skills specification. (2026). agentskills.io. https://agentskills.io/specification (could not be fetched directly; the constraints above come from a verbatim copy and secondary pages)

**Secondary**

Jisc National Centre for AI. (2026). *Agent Skills (the emerging standard) and how it relates to AI skills.* https://nationalcentreforai.jiscinvolve.org/wp/2026/10/28/agent-skills-the-emerging-standard-and-how-it-relates-to-ai-skills/

Agentailor. (2026). *Top AI Agent Standards to Know in 2026.* https://blog.agentailor.com/posts/top-ai-agent-standards-2026

*Knowledge Activation: AI Skills as the Institutional Knowledge Primitive for Agentic Software Development.* (2026). arXiv:2603.14805. https://arxiv.org/pdf/2603.14805

Developers Digest. (2026). *Securing AI coding agents.* https://www.developersdigest.tech/blog/securing-ai-coding-agents
